# Week 18, Day 3: The bootstrap and confidence intervals

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Bootstrap the median

Write `bootstrap(x, stat, B, rng)` returning B values of `stat` on resamples of x with replacement. Apply it to the median of 30 exponential latencies (mean 50 ms) for `se_boot`, and compare with `se_true`, the standard deviation of the median over 5,000 fresh samples from the true distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1979)

def bootstrap(x, stat, B, rng):
    n = len(x)
    return np.array([...])

x = rng.exponential(50, 30)
boot_meds = bootstrap(x, np.median, 4000, rng)
se_boot = boot_meds.std()
se_true = np.std([np.median(rng.exponential(50, 30)) for _ in range(5000)])
lo, hi = np.percentile(boot_meds, [2.5, 97.5])
print(f"sample median {np.median(x):.1f} ms;  bootstrap SE {se_boot:.2f}, true SE {se_true:.2f};  95% interval {lo:.1f} to {hi:.1f}")

In [ ]:
import numpy as np
def test_boot():
    "bootstrap returns B statistics computed on resamples"
    out = bootstrap(np.arange(10.0), np.mean, 500, np.random.default_rng(0))
    assert out.shape == (500,) and 2 < out.mean() < 7

def test_se_close():
    "the bootstrap standard error of the median is within 35% of its true sampling standard deviation"
    assert abs(se_boot - se_true) / se_true < 0.35

test_boot()
test_se_close()
print("All checks passed")

### Exercise: How often does the interval contain the truth?

For samples of size 8 and 50 from an exponential distribution with mean 50, compute 400 percentile bootstrap intervals for the mean (B = 1,000 each) and store the fraction that contain 50 in `cov[n]`.

In [ ]:
r4 = np.random.default_rng(8)
cov = {}
for n in (8, 50):
    hits = 0
    for _ in range(400):
        xs = r4.exponential(50, n)
        bm = xs[r4.integers(0, n, (1000, n))].mean(axis=1)     # 1000 bootstrap means at once
        lo, hi = ...
        hits += ...
    cov[n] = hits / 400
print("coverage of 95% percentile intervals:", cov)

In [ ]:
import numpy as np
def test_coverage():
    "with n = 50 the coverage is at least 0.91; with n = 8 it is lower than with n = 50"
    assert cov[50] >= 0.91 and cov[8] < cov[50]

test_coverage()
print("All checks passed")

### Exercise: The hierarchical bootstrap

Each of 5 mice has its own true effect (mean 1, sd 0.5 between mice); each of 40 neurons per mouse adds noise (sd 1). Write `naive_ci(data, B, rng)` (resample all 200 neurons) and `hier_ci(data, B, rng)` (resample mice, then neurons within each chosen mouse), each returning a 95% percentile interval for the mean effect. Compare widths on one dataset, then coverage of the true mean 1 over 150 datasets.

In [ ]:
r5 = np.random.default_rng(2020)
def make_data(rng, mice=5, per=40):
    mouse_effect = 1 + 0.5 * rng.normal(size=mice)
    return mouse_effect[:, None] + rng.normal(size=(mice, per))     # rows are mice

def naive_ci(data, B, rng):
    flat = data.ravel()
    means = flat[rng.integers(0, len(flat), (B, len(flat)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

def hier_ci(data, B, rng):
    m, k = data.shape
    means = np.empty(B)
    for b in range(B):
        mice = ...                                   # m mice drawn with replacement
        rows = data[mice]
        cols = rng.integers(0, k, (m, k))            # neurons drawn within each chosen mouse
        means[b] = ...
    return np.percentile(means, [2.5, 97.5])

data = make_data(r5)
ci_n, ci_h = naive_ci(data, 2000, r5), hier_ci(data, 2000, r5)
width_naive, width_hier = np.diff(ci_n)[0], np.diff(ci_h)[0]
cov_naive = np.mean([(lambda c: c[0] <= 1 <= c[1])(naive_ci(make_data(r5), 400, r5)) for _ in range(150)])
cov_hier = np.mean([(lambda c: c[0] <= 1 <= c[1])(hier_ci(make_data(r5), 400, r5)) for _ in range(150)])
print(f"widths: naive {width_naive:.2f}, hierarchical {width_hier:.2f};  coverage: naive {cov_naive:.2f}, hierarchical {cov_hier:.2f}")

In [ ]:
import numpy as np
def test_widths():
    "on one dataset, the hierarchical interval is more than twice as wide as the naive one"
    assert width_hier > 2 * width_naive

def test_coverage():
    "over 150 datasets, naive intervals cover the truth less than 60% of the time and hierarchical ones more than 75%"
    assert cov_naive < 0.6 and cov_hier > 0.75

test_widths()
test_coverage()
print("All checks passed")